# Aligning Async Streams

Real-world data sources tick at their own rate: one feed might update every
second, another every five minutes.  Aligning them by row number is wrong -
it implicitly assumes they tick in sync.  screamer's multi-stream layer uses
an explicit **order-key** (an integer timestamp or sequence number) so that
`combine_latest` can forward-fill each input's last seen value and emit one
aligned row whenever any input ticks.  The carry is fully causal: it
propagates the last *observed* value, never a future one.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import combine_latest, merge

# Two async feeds with different timestamps (keys)
a_k = np.array([1, 3, 5, 7, 9], dtype=np.int64)
a_v = np.array([10., 11., 12., 11., 13.])

b_k = np.array([2, 4, 6, 8], dtype=np.int64)
b_v = np.array([5., 6., 5.5, 6.5])

## The order-key model

A **stream** in screamer is a pair `(keys, values)` where `keys` is a
monotonically non-decreasing `int64` array.  Keys represent logical time
(nanoseconds, sequence numbers, bar indices - any metric).  Operations
respect keys rather than row positions, so two streams can be combined
correctly even when their tick rates differ.

In [ ]:
print("Feed a - keys:", a_k, "  values:", a_v)
print("Feed b - keys:", b_k, "  values:", b_v)
print()
print("Feed a ticks at odd keys; feed b at even keys - no shared timestamps.")
print("Row-number alignment would pair a[0]=10 with b[0]=5, but they are at")
print("keys 1 and 2 - a full tick apart.")

## `combine_latest` as-of join

`combine_latest` merges N streams by forward-filling each input's last known
value and emitting one aligned row on every tick.  The default
`emit="when_all"` waits until every input has been seen at least once before
producing output, so the first row always has a real observation from every
feed - no cold-start NaNs.  Once aligned, the two columns can be fed
directly into any functor - in eager array code, unpack first:

```python
keys, aligned = combine_latest((a_k, a_v), (b_k, b_v))
result = RollingCorr(10)(aligned)     # aligned is the (T, 2) values array
```

(Inside a `Dag`, `combine_latest` returns a Node, so the shorter
`RollingCorr(10)(combine_latest(a, b))` form works there.)

In [ ]:
# combine_latest: emit whenever EITHER ticks, carrying each input's last value
keys, aligned = combine_latest((a_k, a_v), (b_k, b_v))  # emit="when_all" default
spread = aligned[:, 0] - aligned[:, 1]

print("Aligned keys :", keys)
print("aligned[:, 0] (feed a carried):", aligned[:, 0])
print("aligned[:, 1] (feed b carried):", aligned[:, 1])
print("spread (a - b)               :", spread)

# Spread is causal: each spread value uses only events observed so far
assert len(keys) == 8   # 5 ticks from a + 4 ticks from b - 1 skipped (before warm-up)
assert not np.isnan(spread).any(), "when_all guarantees no NaN in spread"

## `emit="when_all"` vs `"on_any"`

`emit="on_any"` starts emitting from the very first event.  Feeds not yet
seen carry `NaN` until their first observation arrives.  This is useful when
you need the earliest possible output and can handle the initial NaN period
(e.g. via `dropna` downstream).

In [ ]:
# on_any emits from the first event (NaN for not-yet-seen inputs)
k2, a2 = combine_latest((a_k, a_v), (b_k, b_v), emit="on_any")

print("on_any keys :", k2)
print("on_any aligned:")
print(a2)

# First row: only feed a has ticked; feed b is NaN
assert np.isnan(a2[0, 1]), "feed b NaN before its first tick"
# After key=2, both feeds have ticked - no more NaN
assert not np.isnan(a2[2:]).any()

# when_all produces a strict subset of on_any (skips the cold-start rows)
np.testing.assert_array_equal(keys, k2[1:])    # when_all drops key=1
np.testing.assert_array_equal(aligned, a2[1:]) # values are the same after warm-up

## `merge` - one sorted tagged stream

`merge` interleaves N streams into a single key-sorted `(keys, values,
sources)` triple.  The `sources` array records which input each event came
from.  This is the building block for `pace` (notebook 08) and `split`
(notebook 09); it is also useful for feeding heterogeneous events into a
single downstream processor.

In [ ]:
# merge: interleave into one key-sorted, source-tagged stream
mk, mv, ms = merge((a_k, a_v), (b_k, b_v))

print("merged keys   :", mk)
print("merged values :", mv)
print("sources (0=a, 1=b):", ms)

assert (np.diff(mk) >= 0).all(), "merge output is globally key-sorted"
assert len(mk) == len(a_k) + len(b_k), "merge concatenates all events"

## Plotting the aligned spread

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

# Raw feeds
axes[0].step(a_k, a_v, where="post", label="feed a", color="steelblue")
axes[0].step(b_k, b_v, where="post", label="feed b", color="coral")
axes[0].set_ylabel("price")
axes[0].legend()
axes[0].set_title("Raw async feeds")

# Aligned spread
axes[1].step(keys, spread, where="post", label="spread (a−b, when_all)", color="purple")
axes[1].axhline(0, color="gray", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("key (logical time)")
axes[1].set_ylabel("spread")
axes[1].legend()
axes[1].set_title("Spread computed on aligned values")

plt.tight_layout()
plt.show()

**Takeaways**

- A stream is `(keys_int64, values_float64)`; keys are logical time, not row
  indices.  Alignment respects keys, so different tick rates are handled
  automatically.
- `combine_latest` is a causal as-of join: it forward-fills each input's last
  *observed* value - no lookahead, no future information.
- `emit="when_all"` (default) suppresses output until every feed has at least
  one observation; `emit="on_any"` emits from the first tick with NaN for
  cold inputs.
- `merge` interleaves N streams into one sorted tagged stream; `split` inverts
  it (see notebook 09).
- The aligned columns can be passed directly to any functor - unpack first in
  eager code: `keys, aligned = combine_latest((a_k, a_v), (b_k, b_v));`
  `RollingCorr(10)(aligned)` gives a rolling correlation between two async
  feeds with no extra wiring.  (Inside a `Dag` the shorter
  `RollingCorr(10)(combine_latest(a, b))` form works because `combine_latest`
  there returns a Node.)